# 06 - Out-of-Sample Showdown

A rolling-window backtest comparing five strategies on synthetic monthly returns: plug-in MVO, MVO with Ledoit-Wolf shrinkage, minimum-variance with shrinkage, Black-Litterman, and the naive $1/N$ rule of DeMiguel et al. (2009).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from markowitz import BlackLitterman, LedoitWolfCovariance, MeanVariance

rng = np.random.default_rng(2026)
n, T_total, window = 12, 360, 120

mu_true = rng.uniform(0.04, 0.10, size=n) / 12  # monthly
A = rng.standard_normal((n, n))
Sigma_true = (A @ A.T / n + 0.02 * np.eye(n)) / 12
L = np.linalg.cholesky(Sigma_true)
R = mu_true + rng.standard_normal((T_total, n)) @ L.T

## Strategy functions

In [ ]:
def w_ew(R_win):
    return np.full(R_win.shape[1], 1.0 / R_win.shape[1])


def w_mv_sample(R_win, gamma=3.0):
    mu = R_win.mean(axis=0)
    S = np.cov(R_win, rowvar=False, ddof=1)
    return MeanVariance(risk_aversion=gamma, long_only=True).fit(mu, S).weights_


def w_mv_lw(R_win, gamma=3.0):
    mu = R_win.mean(axis=0)
    S = LedoitWolfCovariance().fit(R_win).covariance_
    return MeanVariance(risk_aversion=gamma, long_only=True).fit(mu, S).weights_


def w_min_var_lw(R_win):
    S = LedoitWolfCovariance().fit(R_win).covariance_
    mu_zero = np.zeros(R_win.shape[1])
    return MeanVariance(risk_aversion=1e6, long_only=True).fit(mu_zero, S).weights_


def w_bl(R_win, delta=2.5, tau=0.05):
    n_ = R_win.shape[1]
    S = LedoitWolfCovariance().fit(R_win).covariance_
    w_mkt = np.full(n_, 1.0 / n_)
    pi = delta * S @ w_mkt
    P = np.eye(n_)[:1]
    q = np.array([R_win[:, 0].mean()])
    bl = BlackLitterman(prior_returns=pi, sigma=S, P=P, q=q, tau=tau, omega="he-litterman").fit()
    return (
        MeanVariance(risk_aversion=delta, long_only=True)
        .fit(bl.posterior_mean_, bl.posterior_cov_)
        .weights_
    )

## Rolling backtest

At each month $t \ge 120$, fit moments on the trailing 120 months, compute weights, hold for the next month, record the realized return.

In [ ]:
strategies = {
    "1/N": w_ew,
    "MVO-sample": w_mv_sample,
    "MVO-LW": w_mv_lw,
    "MinVar-LW": w_min_var_lw,
    "BL": w_bl,
}

oos_returns = {name: [] for name in strategies}
for t in range(window, T_total - 1):
    R_win = R[t - window : t]
    r_next = R[t + 1]
    for name, fn in strategies.items():
        w = fn(R_win)
        oos_returns[name].append(float(w @ r_next))

oos = {k: np.asarray(v) for k, v in oos_returns.items()}

## Summary statistics

In [ ]:
ann = 12  # months -> years
print(f"{'strategy':<14}{'ann.return':>12}{'ann.vol':>10}{'Sharpe':>10}")
for name, r in oos.items():
    mean_a = r.mean() * ann
    vol_a = r.std(ddof=1) * np.sqrt(ann)
    sharpe = mean_a / vol_a if vol_a > 0 else np.nan
    print(f"{name:<14}{mean_a:>12.4f}{vol_a:>10.4f}{sharpe:>10.3f}")

## Cumulative wealth plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, r in oos.items():
    ax.plot(np.cumprod(1.0 + r), label=name, lw=1.5)
ax.set_xlabel("month (out-of-sample)")
ax.set_ylabel("cumulative wealth, $1 initial")
ax.set_title("Out-of-sample horse race")
ax.legend()
fig.tight_layout()

## Reading the result

On this synthetic universe the rankings depend on the seed, but the qualitative pattern is robust:

- Plug-in MVO has the highest turnover and rarely the highest Sharpe.
- LW shrinkage strictly dominates the sample covariance.
- $1/N$ is genuinely hard to beat without informative views.

These conclusions echo DeMiguel, Garlappi & Uppal (2009).